# 5장. 여러 관점으로 보기: 멀티헤드 어텐션

어텐션 헤드 여러 개를 나란히 두고 결과를 합쳐요. 이 장은 **GPU가 필요 없어요.** (런타임 유형: CPU)


### 준비
4장의 결과물 `Head`를 패키지에서 불러옵니다. 이번에는 헤드를 4개 쓰도록 `n_head=4`로 정해요.


In [ ]:
%pip install -q git+https://github.com/yhlee1627/modu-llm

import torch
from torch import nn
from torch.nn import functional as F
from modu_llm import Vocab, GPTConfig, Embeddings, Head, show_weights

torch.manual_seed(42)
vocab = Vocab.build()
cfg = GPTConfig(vocab_size=len(vocab), block_size=16, n_embd=16, n_head=4)
emb = Embeddings(cfg)

sentence = "토끼가 숲에 가요"
labels = list(sentence.replace(" ", "␣"))   # 띄어쓰기를 눈에 보이게
idx = torch.tensor([vocab.encode(sentence)])
x = emb(idx)
print("x 모양:", x.shape)


## 5.1 헤드가 여러 개 필요한 이유

다음 글자를 맞히려면 여러 가지가 동시에 궁금합니다. 바로 앞 글자, 문장의 말투, 주인공 같은 것들이요.
헤드 하나는 비율 표 하나만 만들므로, **헤드를 여러 개 두어 각자 다른 것을 보게** 해요. 무엇을 볼지는 정해 주지 않아도 학습하며 스스로 나뉩니다.


## 5.2 헤드를 나란히 두고 이어 붙이기

벡터 길이 16을 헤드 4개가 나눠 가져, 헤드마다 길이 4의 결과를 냅니다. 네 결과를 **이어 붙이면** 다시 길이 16이 되고,
마지막 Linear(`proj`)가 여러 헤드의 정보를 한데 섞어요. 먼저 이어 붙이기(`torch.cat`)를 작은 텐서로 봅시다.

**핵심 코드**: `out = torch.cat(outs, dim=-1)`


In [ ]:
a = torch.tensor([1, 2])
b = torch.tensor([3, 4])
print("stack:", torch.stack([a, b]))
print("cat:", torch.cat([a, b]))


In [ ]:
class MultiHead(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        head_size = cfg.n_embd // cfg.n_head
        self.heads = nn.ModuleList()
        for i in range(cfg.n_head):
            self.heads.append(Head(cfg, head_size))
        self.proj = nn.Linear(cfg.n_embd, cfg.n_embd)

    def forward(self, x):
        outs = []
        for h in self.heads:
            outs.append(h(x))
        out = torch.cat(outs, dim=-1)
        out = self.proj(out)
        return out

mh = MultiHead(cfg)
print("헤드 수:", len(mh.heads))


헤드마다 결과를 하나씩 꺼내 모양을 확인하고, 이어 붙여 봅니다.


In [ ]:
outs = []
for i, h in enumerate(mh.heads):
    o = h(x)
    print(f"헤드 {i} 결과:", o.shape)
    outs.append(o)

joined = torch.cat(outs, dim=-1)
print("이어 붙인 뒤:", joined.shape)


## 5.3 들어간 모양 그대로 나오는 출력

멀티헤드 어텐션은 입력과 **같은 모양**을 내놓아요. 그래서 6장에서 이 층을 여러 번 쌓을 수 있습니다.

**핵심 코드**: `out.shape == x.shape`


In [ ]:
out = mh(x)
print("입력:", x.shape)
print("출력:", out.shape)
print("같은가요?", out.shape == x.shape)


## 놀이터

본 모델을 건드리지 않는 활동입니다. 건너뛰어도 다음 장에 영향이 없습니다.

헤드 두 개의 비율 표를 나란히 보세요. 학습 전이라 무작위지만, 헤드마다 표가 다르다는 것은 확인할 수 있어요.
헤드 수를 바꾸면 헤드 하나가 맡는 길이(`head_size`)가 어떻게 바뀌는지도 보세요.


In [ ]:
def head_weights(h, x):
    T = x.shape[1]
    q = h.query(x)
    k = h.key(x)
    scores = q @ k.transpose(-2, -1) / k.shape[-1] ** 0.5
    mask = h.mask[:T, :T]
    scores = scores.masked_fill(mask == 0, float("-inf"))
    return F.softmax(scores, dim=-1)

w0 = head_weights(mh.heads[0], x)
w1 = head_weights(mh.heads[1], x)
display(show_weights(w0, labels))
display(show_weights(w1, labels))


In [ ]:
for n in [1, 2, 4, 8]:
    size = cfg.n_embd // n
    print(f"헤드 {n}개 → 헤드 하나의 길이 {size}")


## 실제 LLM에서는

- 실제 LLM은 헤드가 수십 개이고, 헤드 하나의 길이는 보통 64~128이에요.
- 메모리를 아끼려고 여러 헤드가 Key와 Value를 함께 쓰게 하는 방식(그룹 쿼리 어텐션 등)이 널리 쓰입니다.
- 이 장의 코드는 헤드를 반복문으로 하나씩 계산해요. 실제 모델은 모든 헤드를 한 번에 계산하는데, 8장에서 같은 결과를 내는 빠른 방법으로 바꿔 봅니다.
